In [ ]:
### The diagnostic

### DATES CONVERSION
# while reading
    # df = pd.read_csv("path", parse_dates=["date"])
# or afterwards
    # df["date"] = pd.to_datetime(df["date"])
# Once it is in a date format, to extract years, month etc:
    # df["date"].dt.year 
    # df["date"].dt.month 
    # df["date"].dt.to_period("M") (equivalent modf(date))
# For comparison: df["date"] >= "2025-09-16"

import pandas as pd 
df = pd.read_parquet("../data/clean/listings_clean.parquet")

# 1. Load and inspect the clean table: number of rows and columns, column types, first 5 rows.
display(df.shape)
display(df.dtypes)
display(df.head(5))

# 2. Missing values: the number of missing values in each column.
display(
    df
    .isna()
    .sum()
)

# 3. Group statistics: the median price_eur by room_type.
display(
    df
    .groupby("room_type", as_index=False)
    .agg(median=("price_eur","median"))
)

# 4. Filter and sort: priced entire homes in Élysée for at least 4 guests. How many are there? Show the 5 most expensive.
# Create the conditions in one line: 
mask=((df["room_type"]=="Entire home/apt") & (df["accommodates"]>=4) & (df["neighbourhood_cleansed"]=="Élysée") & (df["price_eur"].notna()))
# How many are there:
display(
    df["id"]
    .loc[mask]
    .count()
)
# display 5 most expensive
display(
    df
    .loc[mask]
    .sort_values("price_eur", ascending=False)
    .head(5)
)

# 5. New column: create price_per_guest (price divided by accommodates), then show the 3 neighbourhoods with the highest median price per guest.
df["price_per_guest"] = df["price_eur"]/df["accommodates"]
display(
    df
    .groupby("neighbourhood_cleansed", as_index=False)
    .agg(median=("price_per_guest","median"))
    .sort_values("median", ascending=False)
    .head(3)
)

# 6. Dates: load ../data/raw/reviews.csv, make sure date is a date type, and count reviews per year for 2023 to 2025.
df_reviews = pd.read_csv("../data/raw/reviews.csv", parse_dates=["date"])
assert df_reviews["date"].dtypes == 'datetime64[us]'
df_reviews["year"]=df_reviews["date"].dt.year
display(
    df_reviews
    .loc[(df_reviews["year"]>=2023) & (df_reviews["year"]<=2025)]
    .groupby("year", as_index=False)
    .agg(nb_review=("listing_id","count"))
)

# 7. Merge: add each listing’s number of reviews (from the reviews file) to the listings table, with 0 for listings never reviewed, 
# then compute the mean number of reviews by room_type. You’ve done this in SQL; here, the question is how pandas does it.

df_reviews_tomerge=(
    df_reviews.groupby("listing_id",as_index=False).agg(nb_reviews=("listing_id","count"))
)
df_reviews_tomerge.rename(columns={"listing_id":"id"}, inplace=True)
join = pd.merge(
    df,
    df_reviews_tomerge,
    on="id",
    how='left',
    indicator="merge",
    validate="many_to_one"
)
join["nb_reviews"]=join["nb_reviews"].fillna(0)
display(
    join 
    .groupby("room_type", as_index=False)
    .agg(avg=("nb_reviews","mean"))
)

# 8. reviews per month in 2025
display(
    df_reviews
    .loc[df_reviews["year"]==2025]
    .groupby(df_reviews["date"].dt.month, as_index=False)
    .agg(nb_reviews = ("listing_id","count"))
)


In [ ]:
# Question: Among priced listings that are entire homes in Élysée and have accommodates ≥ 4:
# How many are there?
# Which are the 5 most expensive?

import pandas as pd 
df = pd.read_parquet("../data/clean/listings_clean.parquet")

# Check exact labels
display(df["room_type"].unique())
display(df["neighbourhood_cleansed"].unique())

# Set conditions
mask = (
    (df["neighbourhood_cleansed"]=="Élysée") & 
    (df["room_type"]=="Entire home/apt") & 
    (df["accommodates"]>=4) &
    (df["price_eur"].notna())
)

# How many are there?
print(mask.sum())

# Which are the 5 most expensive?
display(
    df
    .loc[mask,["id","neighbourhood_cleansed","room_type","accommodates", "price_eur"]]
    .sort_values("price_eur", ascending=False)
    .head(5)
)

# Sanity checks:
#   1. Is your count smaller than the number of entire homes in Élysée with no other filter?
print( ((df["neighbourhood_cleansed"]=="Élysée") & (df["room_type"]=="Entire home/apt")).sum() ) # 2,276 -> yes, 1,005 is smaller
#   2. Yes it returns 0